# CANDID-PTX Novelties Pipeline
Semi-supervised pneumothorax segmentation — SIIM-ACR dataset.
**Drive-persistent**: every step auto-saves/restores from Drive. Safe to restart anytime.


## Section 1 — Mount Drive & Define Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os

DRIVE_BASE   = '/content/drive/MyDrive/ptx_pipeline'
DRIVE_DATA   = f'{DRIVE_BASE}/inputs'
DRIVE_MODELS = f'{DRIVE_BASE}/models'
DRIVE_WL     = f'{DRIVE_BASE}/weak_labels'
DRIVE_MEDSAM = '/content/drive/MyDrive/MEDSAM/medsam_vit_b.pth'
LOCAL_INPUTS = '/content/Auto-Generate-WLs/pytorch-nested-unet/inputs'

for d in [DRIVE_DATA, DRIVE_MODELS, DRIVE_WL]:
    os.makedirs(d, exist_ok=True)

assert os.path.exists(DRIVE_MEDSAM), f'Put medsam_vit_b.pth at {DRIVE_MEDSAM}'
print('✅ Drive mounted.')
print(f'Pipeline root: {DRIVE_BASE}')

## Section 2 — Clone Repos & Install

In [ ]:
%cd /content
!rm -rf Auto-Generate-WLs MedSAM
!git clone --quiet https://github.com/stanfordmlgroup/Auto-Generate-WLs

!pip install -q segment-anything albumentations==1.3.1 \
    scikit-image scikit-learn tqdm pydicom kagglehub

# Fix albumentations import breakage in cloned code
import subprocess
for pat in [
    r's/from albumentations.augmentations import transforms/import albumentations as transforms/g',
    r's/from albumentations.core.composition import Compose, OneOf/from albumentations import Compose, OneOf/g',
    r's/from albumentations.core.composition import Compose/from albumentations import Compose/g',
]:
    subprocess.run(["find", "/content/Auto-Generate-WLs/pytorch-nested-unet",
                    "-name", "*.py", "-exec", "sed", "-i", pat, "{}", "+"])

os.makedirs(LOCAL_INPUTS, exist_ok=True)
print('✅ Done.')

## Section 3 — Verify Environment

In [ ]:
import albumentations as A, numpy as np, cv2, torch
print('albumentations:', A.__version__)
print('torch         :', torch.__version__)
print('CUDA          :', torch.cuda.is_available())
dummy = np.zeros((64,64,3), dtype=np.uint8)
_ = A.Compose([A.HorizontalFlip(p=1.0)])(image=dummy)
print('✅ Albumentations OK')

## Section 4 — Download Dataset (YOU MUST RUN THIS CELL)
Even if it says "auto-skipped", **you still have to click the run button**. It will automatically restore data from Drive if it exists.


In [ ]:
import os, shutil

# ── FAST PATH: restore from Drive if already processed ───────────────────────
gs_drive = os.path.join(DRIVE_DATA, 'ptx-gs-25', 'images')
if os.path.exists(gs_drive) and len(os.listdir(gs_drive)) >= 25:
    print('✅ Drive backup found — restoring data...')
    for sn in ['ptx-gs-25', 'ptx-unlabeled', 'ptx-test']:
        dst = os.path.join(LOCAL_INPUTS, sn)
        src = os.path.join(DRIVE_DATA, sn)
        if os.path.exists(dst): shutil.rmtree(dst) # force clean copy
        if os.path.exists(src):
            shutil.copytree(src, dst)
        n = len(os.listdir(os.path.join(dst,'images'))) if os.path.exists(os.path.join(dst,'images')) else 0
        print(f'  {sn}: {n} images')
    print('\n✅ Data restored. Jump to Section 5.')
else:
    print('❌ No Drive backup. Run the next download cell below.')

In [ ]:
# ── ONLY run this if the cell above said "No Drive backup" ───────────────────
import os, glob, shutil, random, cv2, numpy as np, pandas as pd
import pydicom

# ─── PASTE YOUR KAGGLE API TOKEN HERE ────────────────────────────────────────
KAGGLE_API_TOKEN = 'PASTE_YOUR_KGAT_TOKEN_HERE'
# ─────────────────────────────────────────────────────────────────────────────
assert KAGGLE_API_TOKEN != 'PASTE_YOUR_KGAT_TOKEN_HERE', \
    'Paste your KGAT token first!'

os.environ['KAGGLE_API_TOKEN'] = KAGGLE_API_TOKEN
print(f'Token: {KAGGLE_API_TOKEN[:15]}...')

import kagglehub
print('Downloading (412 MB — ~30 sec on Colab)...')
dl_path = kagglehub.competition_download('siim-acr-pneumothorax-segmentation')
print(f'Downloaded to: {dl_path}')

# Copy to siim_raw
SIIM_RAW = '/content/siim_raw'
os.makedirs(SIIM_RAW, exist_ok=True)
for item in os.listdir(dl_path):
    s = os.path.join(dl_path, item); d = os.path.join(SIIM_RAW, item)
    if os.path.isdir(s): shutil.copytree(s, d, dirs_exist_ok=True)
    else: shutil.copy2(s, d)
print('Copied to /content/siim_raw')

# ── Parse CSV (SIIM stage-2 format) ──────────────────────────────────────────
CSV_PATH  = '/content/siim_raw/stage_2_train.csv'
DCM_DIR   = '/content/siim_raw/stage_2_images'

df = pd.read_csv(CSV_PATH)
print(f'CSV rows: {len(df)}  columns: {list(df.columns)}')

ID_COL  = 'ImageId'
RLE_COL = ' EncodedPixels'   # leading space is intentional (SIIM format)

# Fallback: auto-detect columns
if ID_COL not in df.columns:
    ID_COL = df.columns[0]
if RLE_COL not in df.columns:
    RLE_COL = [c for c in df.columns if 'pixel' in c.lower() or 'rle' in c.lower() or 'encoded' in c.lower()][0]
print(f'Using id_col="{ID_COL}"  rle_col="{RLE_COL}"')

df[RLE_COL] = df[RLE_COL].astype(str).str.strip()
df_pos = df[~df[RLE_COL].isin(['-1','nan',''])].copy()
df_neg = df[ df[RLE_COL].isin(['-1','nan',''])].copy()

all_dcm = glob.glob(os.path.join(DCM_DIR, '**', '*.dcm'), recursive=True)
print(f'DICOM files found: {len(all_dcm)}')

# Robust mapping: map both filename and SOPInstanceUID
dcm_map = {}
for p in all_dcm:
    bname = os.path.splitext(os.path.basename(p))[0]
    dcm_map[bname] = p
    try:
        ds = pydicom.dcmread(p, stop_before_pixels=True)
        if hasattr(ds, 'SOPInstanceUID'):
            dcm_map[str(ds.SOPInstanceUID)] = p
    except:
        pass

random.seed(42)
pos_ids = [i for i in df_pos[ID_COL].unique() if i in dcm_map]
neg_ids = [i for i in df_neg[ID_COL].unique() if i in dcm_map]

if len(pos_ids) < 75:
    raise RuntimeError(f"❌ Error: Not enough positive images found (found {len(pos_ids)}). ID matching failed.\n"
                       f"CSV IDs sample: {list(df[ID_COL].unique()[:5])}\n"
                       f"DICOM Map keys sample: {list(dcm_map.keys())[:5]}")

random.shuffle(pos_ids); random.shuffle(neg_ids)
print(f'Positive: {len(pos_ids)}  Negative: {len(neg_ids)}')

test_ids      = pos_ids[:50]
gs_ids        = pos_ids[50:75]
unlabeled_ids = (pos_ids[75:] + neg_ids)[:200]

# Sanity check
assert not (set(test_ids) & set(gs_ids) & set(unlabeled_ids)), 'Leakage!'
print(f'Test:{len(test_ids)}  GS:{len(gs_ids)}  Unlabeled:{len(unlabeled_ids)} ✅')

# ── RLE helpers ───────────────────────────────────────────────────────────────
def rle2mask(rle, w=1024, h=1024):
    mask = np.zeros(w*h, dtype=np.uint8)
    if str(rle).strip() in ['-1','nan','']: return mask.reshape(h,w)
    try:
        s = list(map(int, str(rle).split()))
        for st, ln in zip(s[0::2], s[1::2]): mask[st:st+ln] = 255
    except: pass
    return mask.reshape(h,w).T

def merge_masks(rle_list, size=256):
    m = np.zeros((1024,1024), dtype=np.uint8)
    for r in rle_list: m = np.maximum(m, rle2mask(r))
    return cv2.resize(m, (size,size))

def dcm_to_png(dcm_path, out_path, size=256):
    try:
        ds = pydicom.dcmread(dcm_path)
        arr = ds.pixel_array.astype(np.float32)
        arr = ((arr-arr.min())/(arr.max()-arr.min()+1e-8)*255).astype(np.uint8)
        rgb = cv2.cvtColor(arr, cv2.COLOR_GRAY2BGR)
        cv2.imwrite(out_path, cv2.resize(rgb,(size,size))); return True
    except: return False

rle_by_id = {}
for _, row in df_pos.iterrows():
    rle_by_id.setdefault(row[ID_COL], []).append(str(row[RLE_COL]))

# ── Convert & save splits ─────────────────────────────────────────────────────
for sn in ['ptx-gs-25','ptx-unlabeled','ptx-test']:
    for sub in ['images','masks/0']:
        os.makedirs(os.path.join(LOCAL_INPUTS, sn, sub), exist_ok=True)

ok = fail = 0
for sn, ids in [('ptx-gs-25',gs_ids),('ptx-unlabeled',unlabeled_ids),('ptx-test',test_ids)]:
    img_out  = os.path.join(LOCAL_INPUTS, sn, 'images')
    mask_out = os.path.join(LOCAL_INPUTS, sn, 'masks', '0')
    has_mask = sn != 'ptx-unlabeled'
    for iid in ids:
        dcm = dcm_map.get(iid)
        if not dcm: fail+=1; continue
        if not dcm_to_png(dcm, os.path.join(img_out, iid+'.png')): fail+=1; continue
        if has_mask:
            cv2.imwrite(os.path.join(mask_out, iid+'.png'),
                        merge_masks(rle_by_id.get(iid,['-1'])))
        ok += 1

print(f'Converted: {ok}  Failed: {fail}')
for sn in ['ptx-gs-25','ptx-unlabeled','ptx-test']:
    n = len(os.listdir(os.path.join(LOCAL_INPUTS,sn,'images')))
    print(f'  {sn}: {n} images')

if ok < 100:
    raise RuntimeError(f"❌ Failed to convert enough images (only {ok} converted). Please check logs.")

# ── Save to Drive immediately ─────────────────────────────────────────────────
print('\nSaving to Drive...')
for sn in ['ptx-gs-25','ptx-unlabeled','ptx-test']:
    src = os.path.join(LOCAL_INPUTS, sn)
    dst = os.path.join(DRIVE_DATA, sn)
    if os.path.exists(dst): shutil.rmtree(dst)
    shutil.copytree(src, dst)
    print(f'  ✅ {sn} saved to Drive')
print('Done — future restarts restore from Drive automatically.')

## Section 5 — Contrastive Pretraining (Idea 8)

In [ ]:
import torch, torch.nn as nn, torch.optim as optim
import torchvision.transforms as T
from torch.utils.data import Dataset as TDS, DataLoader
from PIL import Image
import os, numpy as np

class ContrastiveTransform:
    def __init__(self, sz=128):
        self.t = T.Compose([T.RandomResizedCrop(sz,scale=(0.5,1.0)),
            T.RandomHorizontalFlip(), T.ColorJitter(0.3,0.3,0.1,0.05),
            T.ToTensor(), T.Normalize([0.5]*3,[0.5]*3)])
    def __call__(self, x): return self.t(x), self.t(x)

class UDS(TDS):
    def __init__(self, d, tf):
        self.paths = [os.path.join(d,f) for f in os.listdir(d) if f.endswith('.png')]
        self.tf = tf
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        return self.tf(Image.open(self.paths[i]).convert('RGB'))

class Enc(nn.Module):
    def __init__(self):
        super().__init__()
        self.bb = nn.Sequential(
            nn.Conv2d(3,32,3,2,1),nn.ReLU(),nn.Conv2d(32,64,3,2,1),nn.ReLU(),
            nn.Conv2d(64,128,3,2,1),nn.ReLU(),nn.AdaptiveAvgPool2d((4,4)))
        self.proj = nn.Sequential(nn.Flatten(),nn.Linear(2048,256),nn.ReLU(),nn.Linear(256,128))
    def forward(self, x): return self.proj(self.bb(x))

def nt_xent(z1, z2, temp=0.5):
    z1 = nn.functional.normalize(z1,dim=1); z2 = nn.functional.normalize(z2,dim=1)
    N = z1.size(0); z = torch.cat([z1,z2])
    sim = torch.mm(z,z.T)/temp
    sim.masked_fill_(torch.eye(2*N,dtype=torch.bool,device=z.device), float('-inf'))
    return nn.CrossEntropyLoss()(sim, torch.cat([torch.arange(N,2*N),torch.arange(N)]).to(z.device))

device = 'cuda' if torch.cuda.is_available() else 'cpu'
ul_dir = os.path.join(LOCAL_INPUTS, 'ptx-unlabeled', 'images')

if not os.path.exists(ul_dir):
    print("❌ Dataset missing! Run Section 4 to restore data from Drive.")
else:
    ds = UDS(ul_dir, ContrastiveTransform(128))
    if len(ds) >= 4:
        loader = DataLoader(ds, batch_size=min(32,len(ds)), shuffle=True,
                            num_workers=2, drop_last=True)
        enc = Enc().to(device); opt = optim.Adam(enc.parameters(), lr=3e-4)
        enc.train()
        for ep in range(5):
            tot = 0
            for v1, v2 in loader:
                v1,v2 = v1.to(device), v2.to(device)
                loss = nt_xent(enc(v1), enc(v2))
                opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item()
            print(f'Epoch {ep+1}/5  loss={tot/len(loader):.4f}')
        print('✅ Contrastive pretraining done.')
    else:
        print(f'Only {len(ds)} images — skipping.')

## Section 6 — Patch train.py with --seed Support

In [ ]:
%cd /content/Auto-Generate-WLs/pytorch-nested-unet
import re
with open('train.py','r') as f: tr = f.read()
tr = tr.replace('\r\n','\n')

if '--seed' not in tr:
    tr = tr.replace(
        "parser.add_argument('--num_workers', default=4, type=int)",
        "parser.add_argument('--num_workers', default=4, type=int)\n"
        "    parser.add_argument('--seed', default=42, type=int)")
    seed_block = '''
    import random as _random, numpy as _np, torch as _torch
    _s = config['seed']
    _random.seed(_s); _np.random.seed(_s); _torch.manual_seed(_s)
    if _torch.cuda.is_available():
        _torch.cuda.manual_seed_all(_s)
        _torch.backends.cudnn.deterministic = True
        _torch.backends.cudnn.benchmark = False
'''
    tr = tr.replace('config = vars(parse_args())', 'config = vars(parse_args())' + seed_block)
    with open('train.py','w') as f: f.write(tr)
    print('train.py patched with --seed.')
else:
    print('--seed already present.')

%cd /content

## Section 7 — Train Baseline NestedUNet (auto-skipped if Drive has model)

In [ ]:
import os, shutil

LOCAL_MODEL = '/content/Auto-Generate-WLs/pytorch-nested-unet/models/ptx-gs-25_NestedUNet_256_woDS'
DRIVE_MODEL = os.path.join(DRIVE_MODELS, 'ptx-gs-25_NestedUNet_256_woDS')

if not os.path.exists('/content/Auto-Generate-WLs/pytorch-nested-unet/inputs/ptx-gs-25/images') or len(os.listdir('/content/Auto-Generate-WLs/pytorch-nested-unet/inputs/ptx-gs-25/images')) == 0:
    raise RuntimeError("❌ Dataset not found! You MUST run Section 4 (the download cell) to properly convert the data.")

if os.path.exists(os.path.join(DRIVE_MODEL, 'model.pth')):
    if not os.path.exists(os.path.join(LOCAL_MODEL, 'model.pth')):
        shutil.copytree(DRIVE_MODEL, LOCAL_MODEL)
    print('✅ Baseline model restored from Drive. Skip to Section 8.')
else:
    print('Training baseline (~10 min on T4 GPU)...')
    %cd /content/Auto-Generate-WLs/pytorch-nested-unet
    !python train.py \
        --dataset ptx-gs-25 \
        --name ptx-gs-25_NestedUNet_256_woDS \
        --arch NestedUNet \
        --img_ext .png --mask_ext .png \
        --input_channels 3 \
        --batch_size 4 --input_w 256 --input_h 256 \
        --seed 42 --optimizer Adam --lr 3e-4 --epochs 200
    %cd /content
    if os.path.exists(os.path.join(LOCAL_MODEL, 'model.pth')):
        if os.path.exists(DRIVE_MODEL): shutil.rmtree(DRIVE_MODEL)
        shutil.copytree(LOCAL_MODEL, DRIVE_MODEL)
        print('✅ Baseline model saved to Drive.')
    else:
        raise RuntimeError('Training failed — check output above for errors.')

## Section 8 — Evaluate Baseline on ptx-test

In [ ]:
%cd /content/Auto-Generate-WLs/pytorch-nested-unet
!python eval.py \
    --name ptx-gs-25_NestedUNet_256_woDS \
    --dataset ptx-test \
    --img_ext .png --mask_ext .png
%cd /content

## Section 9 — Complexity Gate Helper (Idea 1)

In [ ]:
import os

complexity_gate_src = '''
import cv2, numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

def compute_complexity_score(path):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        c = cv2.imread(path)
        img = cv2.cvtColor(c, cv2.COLOR_BGR2GRAY) if c is not None else np.zeros((256,256), np.uint8)
    img = cv2.resize(img, (256, 256))
    n = img / 255.0
    h, _ = np.histogram(n.ravel(), bins=32, density=True); h += 1e-10
    return {
        "local_variance": float(np.std(n)),
        "edge_density":   float(np.sum(cv2.Canny(img,50,150)>0)/(256*256)),
        "entropy":        float(-np.sum(h*np.log(h))),
        "bg_homogeneity": float(1.0/(np.std(np.concatenate([n[0,:],n[-1,:],n[:,0],n[:,-1]]))+1e-5)),
    }

def train_complexity_classifier(gs_img, gs_mask, preds):
    X, y = [], []
    for ip, gp, pm in zip(gs_img, gs_mask, preds):
        X.append(list(compute_complexity_score(ip).values()))
        gt = (cv2.imread(gp, cv2.IMREAD_GRAYSCALE) > 127).astype(np.uint8)
        pr = (pm > 127).astype(np.uint8)
        inter = np.logical_and(gt, pr).sum()
        y.append(1 if 2.*inter/(gt.sum()+pr.sum()+1e-6) > 0.5 else 0)
    sc = StandardScaler(); Xs = sc.fit_transform(X)
    clf = RandomForestClassifier(n_estimators=50, random_state=42)
    clf.fit(Xs, y); return clf, sc

def gate_images(paths, clf, sc, threshold=0.5):
    approved, rejected = [], []
    for p in paths:
        try:
            f = sc.transform([list(compute_complexity_score(p).values())])
            prob = clf.predict_proba(f)[0][1] if len(clf.classes_) >= 2 else float(clf.classes_[0]==1)
            (approved if prob >= threshold else rejected).append(p)
        except:
            rejected.append(p)
    if len(approved) < 40:
        def prob_fn(p):
            try:
                f = sc.transform([list(compute_complexity_score(p).values())])
                return clf.predict_proba(f)[0][1] if len(clf.classes_) >= 2 else 1.
            except: return 0.
        ranked = sorted(paths, key=prob_fn, reverse=True)
        approved, rejected = ranked[:40], ranked[40:]
    return approved, rejected
'''

os.makedirs('Auto-Generate-WLs/generate-weak-labels', exist_ok=True)
with open('Auto-Generate-WLs/generate-weak-labels/complexity_gate.py', 'w') as f:
    f.write(complexity_gate_src)
print('complexity_gate.py written.')

## Section 10 — TTA Consistency Check (Idea 3)

In [ ]:
import torch, torchvision.transforms.functional as TF, numpy as np, cv2
from scipy import ndimage as ndi

def augmentation_consistency_check(image, model, device, n_augments=6, iou_threshold=0.20):
    if image.ndim == 2: image = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)
    model.eval(); boxes = []
    small = cv2.resize(image, (256,256))
    t_base = torch.tensor(small.transpose(2,0,1), dtype=torch.float32).unsqueeze(0).to(device)/255.
    for angle in np.linspace(-30,30,n_augments):
        t = TF.rotate(t_base, float(angle))
        with torch.no_grad():
            p = model(t)
            p = p[-1] if isinstance(p,list) else p
            p = torch.sigmoid(p).squeeze().cpu().numpy()
        pb = (p>0.5).astype(np.uint8)
        pr = (ndi.rotate(pb, -float(angle), reshape=False, order=0)>0.5).astype(np.uint8)
        cs,_ = cv2.findContours(pr, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if cs:
            x,y,w,h = cv2.boundingRect(max(cs, key=cv2.contourArea))
            boxes.append([x,y,x+w,y+h])
    if len(boxes) < n_augments//4: return False, 0.
    ious = []
    for i in range(len(boxes)):
        for j in range(i+1,len(boxes)):
            b1,b2 = boxes[i],boxes[j]
            ix = max(0,min(b1[2],b2[2])-max(b1[0],b2[0]))
            iy = max(0,min(b1[3],b2[3])-max(b1[1],b2[1]))
            inter = ix*iy
            union = (b1[2]-b1[0])*(b1[3]-b1[1])+(b2[2]-b2[0])*(b2[3]-b2[1])-inter
            ious.append(inter/(union+1e-5))
    m = float(np.mean(ious)) if ious else 0.
    return m >= iou_threshold, m

## Section 11 — MedSAM Refinement (Idea 5)

In [ ]:
import torch, cv2, numpy as np, os
from segment_anything import sam_model_registry

device = 'cuda' if torch.cuda.is_available() else 'cpu'
medsam_model = sam_model_registry['vit_b'](checkpoint=DRIVE_MEDSAM).to(device)
medsam_model.eval()
print(f'MedSAM loaded on {device}')

def run_medsam_refinement(image_bgr, coarse_mask):
    H,W = image_bgr.shape[:2]
    binary = (coarse_mask>127).astype(np.uint8)
    ys,xs = np.where(binary>0)
    if len(xs)==0: return None
    x1,y1,x2,y2 = xs.min(),ys.min(),xs.max(),ys.max()
    if (x2-x1)<5 or (y2-y1)<5: return None
    scale = 1024/max(H,W)
    H1,W1 = int(round(H*scale)), int(round(W*scale))
    img_rs = cv2.resize(image_bgr,(W1,H1))
    padded = np.zeros((1024,1024,3), dtype=np.uint8); padded[:H1,:W1] = img_rs
    img_f = padded.astype(np.float32)/255.
    img_f = (img_f-img_f.min())/np.clip(img_f.max()-img_f.min(),1e-8,None)
    t = torch.tensor(img_f.transpose(2,0,1)).unsqueeze(0).to(device)
    box = np.clip(np.array([x1*scale,y1*scale,x2*scale,y2*scale],np.float32),0,1023)
    with torch.no_grad():
        emb = medsam_model.image_encoder(t)
        bt  = torch.tensor(box,dtype=torch.float32).to(device).unsqueeze(0).unsqueeze(0)
        se,de = medsam_model.prompt_encoder(points=None,boxes=bt,masks=None)
        logits,_ = medsam_model.mask_decoder(
            image_embeddings=emb, image_pe=medsam_model.prompt_encoder.get_dense_pe(),
            sparse_prompt_embeddings=se, dense_prompt_embeddings=de, multimask_output=False)
    p = torch.sigmoid(logits).cpu().numpy().squeeze()[:H1,:W1]
    return (np.clip(cv2.resize(p,(W,H)),0,1)>0.5).astype(np.uint8)*255

## Section 12 — Morphological Gate V2 (Idea 2) — PTX-tuned

In [ ]:
import cv2 as _cv2, numpy as np

def morphological_gate_v2(coarse_mask):
    import cv2
    mask = (coarse_mask>127).astype(np.uint8)
    H,W = mask.shape[:2]
    fr = (mask>0).mean()
    if fr > 0.90 or fr < 0.001:
        return False, f'extreme_fill({fr:.3f})', None
    nl,labels = cv2.connectedComponents(mask)
    if nl <= 1: return False, 'no_object', None
    sizes = [np.sum(labels==i) for i in range(1,nl)]
    sig   = [s for s in sizes if s >= 30]
    if len(sig) > 300: return False, f'too_fragmented({len(sig)})', None
    cl = (labels==(1+np.argmax(sizes))).astype(np.uint8)
    ss = sorted(sig, reverse=True)
    if len(ss)>=2 and ss[1]/(ss[0]+1e-5)>0.80:
        return False, f'ambiguous({ss[1]/(ss[0]+1e-5):.2f})', None
    cs,_ = cv2.findContours(cl, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cs: return False, 'no_contour', None
    mc = max(cs, key=cv2.contourArea)
    x,y,w,h = cv2.boundingRect(mc)
    ar = max(w,h)/(min(w,h)+1e-5)
    if ar > 25:  return False, f'bad_aspect({ar:.1f})', None
    a = cv2.contourArea(mc)
    if a < 0.0003*H*W: return False, f'too_small({a:.0f})', None
    hull = cv2.convexHull(mc)
    cvx  = a/(cv2.contourArea(hull)+1e-5)
    if cvx < 0.01: return False, f'low_convexity({cvx:.2f})', None
    ys_c,xs_c = np.where(cl>0)
    if len(xs_c)>0 and len(np.unique(xs_c))/(len(np.unique(ys_c))+1e-5)>10:
        return False, 'stripe', None
    return True, 'passed', cl*255

## Section 13 — Radiomics Quality Predictor (Idea 7)

In [ ]:
from skimage.measure import regionprops
from skimage.feature import graycomatrix, graycoprops
from sklearn.ensemble import GradientBoostingRegressor
import cv2, numpy as np

def extract_features(image, mask):
    mb = (mask>127).astype(np.uint8)
    props = regionprops(mb)
    if not props:
        return {'area':0.,'eccentricity':0.,'perimeter':0.,'mean_intensity':0.,'contrast':0.}
    prop = max(props, key=lambda x: x.area)
    ic = (image*255).astype(np.uint8) if image.max()<=1. else image.astype(np.uint8)
    glcm = graycomatrix(ic, distances=[2], angles=[0], levels=256, symmetric=True, normed=True)
    return {
        'area':          float(prop.area),
        'eccentricity':  float(prop.eccentricity),
        'perimeter':     float(prop.perimeter),
        'mean_intensity':float(np.mean(image[mb>0]) if mb.sum()>0 else 0.),
        'contrast':      float(graycoprops(glcm,'contrast')[0,0]),
    }

def train_radiomics_quality_model(gs_imgs, gs_masks, preds):
    X, y = [], []
    for ip,gp,pm in zip(gs_imgs, gs_masks, preds):
        img = cv2.imread(ip, cv2.IMREAD_GRAYSCALE)
        if img is None:
            c = cv2.imread(ip)
            img = cv2.cvtColor(c,cv2.COLOR_BGR2GRAY) if c is not None else np.zeros((256,256),np.uint8)
        gt = (cv2.imread(gp, cv2.IMREAD_GRAYSCALE)>127).astype(np.uint8)
        X.append(list(extract_features(img, pm).values()))
        inter = np.logical_and(gt,(pm>127)).sum()
        y.append(2.*inter/(gt.sum()+(pm>127).sum()+1e-5))
    m = GradientBoostingRegressor(n_estimators=30, random_state=42)
    m.fit(X, y); return m

## Section 14 — Weak Label Generation (auto-skipped if Drive has WLs)
Gates: Complexity → TTA → MedSAM → Morphology → Radiomics. Cap = 100 WLs.

In [ ]:
import sys, os, shutil, torch, cv2, numpy as np
from collections import Counter
from tqdm import tqdm

LOCAL_WL_MASK = os.path.join(LOCAL_INPUTS,'ptx-25-wl-v2','masks','0')
LOCAL_WL_IMG  = os.path.join(LOCAL_INPUTS,'ptx-25-wl-v2','images')
DRIVE_WL_MASK = os.path.join(DRIVE_WL,'masks','0')
DRIVE_WL_IMG  = os.path.join(DRIVE_WL,'images')

# ── Fast path: restore from Drive ────────────────────────────────────────────
if os.path.exists(DRIVE_WL_MASK) and len(os.listdir(DRIVE_WL_MASK))>0:
    n_wl = len(os.listdir(DRIVE_WL_MASK))
    print(f'✅ {n_wl} weak labels on Drive — restoring...')
    for src,dst in [(DRIVE_WL_MASK,LOCAL_WL_MASK),(DRIVE_WL_IMG,LOCAL_WL_IMG)]:
        os.makedirs(dst, exist_ok=True)
        for f in os.listdir(src):
            shutil.copy(os.path.join(src,f), os.path.join(dst,f))
    print('✅ Restored. Skip to Section 15.')
else:
    # ── Load gate helpers ─────────────────────────────────────────────────────
    sys.path.insert(0,'Auto-Generate-WLs/generate-weak-labels')
    import complexity_gate, importlib; importlib.reload(complexity_gate)
    from complexity_gate import train_complexity_classifier, gate_images

    %cd /content/Auto-Generate-WLs/pytorch-nested-unet
    import archs

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    coarse_net = archs.NestedUNet(num_classes=1,input_channels=3,deep_supervision=False).to(device)
    coarse_net.load_state_dict(
        torch.load('models/ptx-gs-25_NestedUNet_256_woDS/model.pth', map_location=device))
    coarse_net.eval(); print(f'Coarse model on {device}')

    GS_IMG  = 'inputs/ptx-gs-25/images'
    GS_MASK = 'inputs/ptx-gs-25/masks/0'
    UL_IMG  = 'inputs/ptx-unlabeled/images'

    gs_imgs  = sorted([os.path.join(GS_IMG,f)  for f in os.listdir(GS_IMG)  if f.endswith('.png')])
    gs_masks = [os.path.join(GS_MASK,os.path.basename(f)) for f in gs_imgs]
    ul_files = sorted([f for f in os.listdir(UL_IMG) if f.endswith('.png')])
    print(f'GS:{len(gs_imgs)}  Unlabeled:{len(ul_files)}')

    def coarse_pred(img_bgr):
        s = cv2.resize(img_bgr,(256,256))
        t = torch.tensor(s.transpose(2,0,1)).unsqueeze(0).float().to(device)/255.
        with torch.no_grad(): p = torch.sigmoid(coarse_net(t)).squeeze().cpu().numpy()
        H,W = img_bgr.shape[:2]
        return (cv2.resize(p,(W,H))>0.5).astype(np.uint8)*255, p

    # Train gate models on GS
    print('Getting GS predictions for gate training...')
    gs_preds = []
    for ip in tqdm(gs_imgs):
        img = cv2.imread(ip)
        if img is None: gs_preds.append(np.zeros((256,256),np.uint8)); continue
        cm,_ = coarse_pred(img)
        ref   = run_medsam_refinement(img, cm)
        gs_preds.append(ref if ref is not None else cm)

    clf_comp,sc_comp  = train_complexity_classifier(gs_imgs, gs_masks, gs_preds)
    qual_reg          = train_radiomics_quality_model(gs_imgs, gs_masks, gs_preds)
    print('Gate models trained.')

    # Coarse-scan unlabeled (memory-safe: delete after scoring)
    print('Coarse-scanning unlabeled images...')
    ul_paths,ul_confs = [],[]
    for fn in tqdm(ul_files):
        ip  = os.path.join(UL_IMG,fn)
        img = cv2.imread(ip)
        if img is None: continue
        s = cv2.resize(img,(256,256))
        t = torch.tensor(s.transpose(2,0,1)).unsqueeze(0).float().to(device)/255.
        with torch.no_grad(): p = torch.sigmoid(coarse_net(t)).squeeze().cpu().numpy()
        ul_paths.append(ip)
        ul_confs.append(p.max() - abs((p>0.5).mean() - 0.10))
        del img,s,t,p

    # Curriculum sort (easiest/highest-confidence first)
    sorted_paths = [p for _,p in sorted(zip(ul_confs,ul_paths),key=lambda x:x[0],reverse=True)]
    del ul_confs

    # Gate 1: Complexity
    approved,rejected = gate_images(sorted_paths, clf_comp, sc_comp, threshold=0.15)
    print(f'Complexity gate: {len(approved)} approved / {len(rejected)} rejected')

    os.makedirs(LOCAL_WL_MASK,exist_ok=True); os.makedirs(LOCAL_WL_IMG,exist_ok=True)
    MAX_WL=100; saved=[]; skipped={}

    for img_path in tqdm(approved, desc='Generating WLs'):
        if len(saved)>=MAX_WL: print(f'✅ Cap {MAX_WL} reached.'); break
        fn  = os.path.basename(img_path)
        img = cv2.imread(img_path)
        if img is None: continue

        cm,_ = coarse_pred(img)

        # Gate 2: TTA
        ok,iou = augmentation_consistency_check(img, coarse_net, device)
        if not ok: skipped[fn]=f'TTA({iou:.2f})'; continue

        # Gate 3: MedSAM
        ref = run_medsam_refinement(img, cm)
        if ref is None: skipped[fn]='medsam_fail'; continue

        # Gate 4: Morphological
        pm,reason,cl = morphological_gate_v2(ref)
        if not pm: skipped[fn]=f'morph_{reason}'; continue

        # Gate 5: Radiomics quality
        gray  = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        feats = extract_features(gray, ref)
        if qual_reg.predict([list(feats.values())])[0] < 0.03:
            skipped[fn]='low_quality'; continue

        cv2.imwrite(os.path.join(LOCAL_WL_MASK,fn), cl)
        shutil.copy(img_path, os.path.join(LOCAL_WL_IMG,fn))
        saved.append(fn)

    print(f'\nSaved {len(saved)} weak labels.')
    print('Skip reasons:', Counter(skipped.values()))

    # Save WLs to Drive
    os.makedirs(DRIVE_WL_MASK,exist_ok=True); os.makedirs(DRIVE_WL_IMG,exist_ok=True)
    for f in saved:
        shutil.copy(os.path.join(LOCAL_WL_MASK,f), os.path.join(DRIVE_WL_MASK,f))
        shutil.copy(os.path.join(LOCAL_WL_IMG,f),  os.path.join(DRIVE_WL_IMG,f))
    print('✅ Weak labels saved to Drive.')

In [ ]:
# Diagnostic
import os
wl = os.path.join(LOCAL_INPUTS,'ptx-25-wl-v2','masks','0')
n  = len(os.listdir(wl)) if os.path.exists(wl) else 0
print(f'Weak labels ready: {n}')

## Section 15 — Distance-Weighted Loss Patch (Idea 6)

In [ ]:
%cd /content/Auto-Generate-WLs/pytorch-nested-unet
import os, subprocess, re

# Reset to clean state then re-apply albumentations fix
os.system('git checkout -- train.py dataset.py losses.py')
for pat in [
    r's/from albumentations.augmentations import transforms/import albumentations as transforms/g',
    r's/from albumentations.core.composition import Compose, OneOf/from albumentations import Compose, OneOf/g',
    r's/from albumentations.core.composition import Compose/from albumentations import Compose/g',
]:
    subprocess.run(["find",".","-name","*.py","-exec","sed","-i",pat,"{}","+"])
print('Git reset + albumentations fix done.')

# Patch dataset.py
with open('dataset.py') as f: ds = f.read()
if 'is_weak' not in ds:
    ds = ds.replace(
        "return img, mask, {'img_id': img_id}",
        "is_weak=1. if 'wl' in img_id or 'weak' in img_id else 0.\n"
        "        return img, mask, {'img_id': img_id, 'is_weak': is_weak}")
    with open('dataset.py','w') as f: f.write(ds)
    print('dataset.py patched.')

# Append DWL + DiceLoss to losses.py
losses_code = '''
import torch as _torch, torch.nn as _nn, numpy as _np
from scipy.ndimage import distance_transform_edt as _dte

class DistanceWeightedLoss(_nn.Module):
    def __init__(self): super().__init__(); self.bce=_nn.BCEWithLogitsLoss(reduction='none')
    def forward(self,out,tgt,wk=None):
        bw=[]
        for i in range(tgt.shape[0]):
            if wk is not None and wk[i]>0.5:
                g=tgt[i,0].detach().cpu().numpy()>0.5
                if g.sum()>0:
                    d=_dte(g)+_dte(~g); d=d/d.max() if d.max()>0 else d; w=0.4+0.6*d
                else: w=_np.ones(g.shape,_np.float32)
                bw.append(_torch.tensor(w,dtype=_torch.float32).unsqueeze(0))
            else: bw.append(_torch.ones(1,*tgt.shape[2:],dtype=_torch.float32))
        pw=_torch.stack(bw).to(out.device)
        bm=self.bce(out,tgt)*pw
        if wk is not None: bm=bm*(1.-0.5*wk).view(-1,1,1,1)
        return bm.mean()

class DiceLoss(_nn.Module):
    def __init__(self): super().__init__()
    def forward(self,inp,tgt,wk=None):
        s=1e-5; inp=_torch.sigmoid(inp); n=tgt.size(0)
        inp=inp.view(n,-1); tgt=tgt.view(n,-1)
        lpi=1-(2.*(inp*tgt).sum(1)+s)/(inp.sum(1)+tgt.sum(1)+s)
        return ((lpi*(1.-0.5*wk)).mean() if wk is not None else lpi.mean())
'''
with open('losses.py','a') as f:
    f.write(losses_code)
print('losses.py patched.')

# Patch train.py robustly
with open('train.py','r') as f: tr = f.read()

# 1. Add val_dataset and seed arguments
tr = re.sub(
    r"(parser\.add_argument\('--num_workers', default=4, type=int\))",
    r"\1\n    parser.add_argument('--seed', default=42, type=int)\n    parser.add_argument('--val_dataset', default=None, type=str)",
    tr
)

# 2. Add random seed setting block
sb='''
    import random as _r,numpy as _n,torch as _t
    _s=config['seed']; _r.seed(_s); _n.random.seed(_s); _t.manual_seed(_s)
    if _t.cuda.is_available(): _t.cuda.manual_seed_all(_s); _t.backends.cudnn.deterministic=True; _t.backends.cudnn.benchmark=False
'''
tr = tr.replace('config = vars(parse_args())', 'config = vars(parse_args())' + sb)

# 3. Patch train_test_split to support --val_dataset (robust regex fix)
patch_tts = '''    img_ids = sorted(img_ids)
    if config.get("val_dataset"):
        train_img_ids = img_ids
        import os
        val_img_ids = sorted([os.path.splitext(fn)[0] for fn in os.listdir(os.path.join("inputs", config["val_dataset"], "images")) if fn.endswith(config["img_ext"])])
        print(f"Train:{len(train_img_ids)} Val:{len(val_img_ids)}")
    else:
        train_img_ids, val_img_ids = train_test_split(img_ids, test_size=0.2, random_state=41)'''

tr = re.sub(r"(\s*img_ids\s*=\s*glob\(.*?\)\s*img_ids\s*=\s*\[.*?\]\s*img_ids\s*=\s*sorted\(img_ids\)\s*)train_img_ids, val_img_ids = train_test_split\(img_ids, test_size=[\d\.]+, random_state=\d+\)", 
            r"\1" + patch_tts.strip(), tr, flags=re.DOTALL)
if 'train_test_split(img_ids' in tr and 'val_dataset' not in tr:
    tr = re.sub(r"train_img_ids, val_img_ids = train_test_split\(img_ids, test_size=[\d\.]+, random_state=\d+\)", patch_tts.strip(), tr)

# 4. Patch DistanceWeightedLoss
if 'DistanceWeightedLoss' not in tr:
    tr=tr.replace('import losses','import losses\nfrom losses import DistanceWeightedLoss,DiceLoss as _DL\n_dwl=DistanceWeightedLoss()')
    tr=tr.replace('for input, target, _ in train_loader:','for input, target, meta in train_loader:')
    tr=tr.replace(
        '        else:\n            output = model(input)\n            loss = criterion(output, target)',
        '        else:\n            output=model(input)\n'
        '            wk=meta.get(\'is_weak\',torch.zeros(target.shape[0])).to(target.device)\n'
        '            loss=_DL()(output,target,wk)+0.5*_dwl(output,target,wk)')
with open('train.py','w') as f: f.write(tr)
print('train.py patched.')
%cd /content

## Section 16 — Build Augmented Dataset & Retrain (auto-skipped if Drive has model)

In [ ]:
import os, shutil

LOCAL_IMP = '/content/Auto-Generate-WLs/pytorch-nested-unet/models/ptx-aug-train_NestedUNet_256_woDS'
DRIVE_IMP = os.path.join(DRIVE_MODELS,'ptx-aug-train_NestedUNet_256_woDS')

if os.path.exists(os.path.join(DRIVE_IMP,'model.pth')):
    if not os.path.exists(os.path.join(LOCAL_IMP,'model.pth')):
        shutil.copytree(DRIVE_IMP, LOCAL_IMP)
    print('✅ Improved model restored from Drive. Skip to Section 17.')
else:
    BASE = LOCAL_INPUTS
    for d in ['ptx-aug-train/images','ptx-aug-train/masks/0',
              'ptx-aug-val/images','ptx-aug-val/masks/0']:
        p=os.path.join(BASE,d)
        if os.path.exists(p): shutil.rmtree(p)
        os.makedirs(p,exist_ok=True)

    gs_files = sorted(os.listdir(os.path.join(BASE,'ptx-gs-25','images')))
    val_files = gs_files[:5]; trn_files = gs_files[5:]

    for f in val_files:
        stem=os.path.splitext(f)[0]
        shutil.copy(os.path.join(BASE,'ptx-gs-25','images',f),
                    os.path.join(BASE,'ptx-aug-val','images',f))
        shutil.copy(os.path.join(BASE,'ptx-gs-25','masks','0',stem+'.png'),
                    os.path.join(BASE,'ptx-aug-val','masks','0',stem+'.png'))
    for f in trn_files:
        stem=os.path.splitext(f)[0]
        shutil.copy(os.path.join(BASE,'ptx-gs-25','images',f),
                    os.path.join(BASE,'ptx-aug-train','images',f))
        shutil.copy(os.path.join(BASE,'ptx-gs-25','masks','0',stem+'.png'),
                    os.path.join(BASE,'ptx-aug-train','masks','0',stem+'.png'))

    wl_mask=os.path.join(BASE,'ptx-25-wl-v2','masks','0')
    wl_img =os.path.join(BASE,'ptx-25-wl-v2','images')
    wl_files=sorted(os.listdir(wl_mask)) if os.path.exists(wl_mask) else []
    for mf in wl_files:
        si=os.path.join(wl_img,mf)
        if not os.path.exists(si): continue
        shutil.copy(si,                       os.path.join(BASE,'ptx-aug-train','images',f'wl_{mf}'))
        shutil.copy(os.path.join(wl_mask,mf), os.path.join(BASE,'ptx-aug-train','masks','0',f'wl_{mf}'))

    n_trn=len(os.listdir(os.path.join(BASE,'ptx-aug-train','images')))
    n_val=len(os.listdir(os.path.join(BASE,'ptx-aug-val','images')))
    print(f'Train:{n_trn} (20 GS + {len(wl_files)} WL)  Val:{n_val} (5 clean GS)')

    %cd /content/Auto-Generate-WLs/pytorch-nested-unet
    !python train.py \
        --dataset ptx-aug-train \
        --val_dataset ptx-aug-val \
        --name ptx-aug-train_NestedUNet_256_woDS \
        --arch NestedUNet \
        --img_ext .png --mask_ext .png \
        --input_channels 3 \
        --batch_size 4 --input_w 256 --input_h 256 \
        --seed 42 --optimizer Adam --lr 3e-4 --epochs 200
    %cd /content

    if os.path.exists(os.path.join(LOCAL_IMP,'model.pth')):
        if os.path.exists(DRIVE_IMP): shutil.rmtree(DRIVE_IMP)
        shutil.copytree(LOCAL_IMP, DRIVE_IMP)
        print('✅ Improved model saved to Drive.')
    else:
        raise RuntimeError('Training failed.')

## Section 17 — Final Evaluation & Comparison

In [ ]:
%cd /content/Auto-Generate-WLs/pytorch-nested-unet
print('='*55)
print('EVALUATION: ptx-test (50 held-out images)')
print('='*55)
print('\n--- 1. BASELINE (25 GS only) ---')
!python eval.py --name ptx-gs-25_NestedUNet_256_woDS --dataset ptx-test --img_ext .png --mask_ext .png
print('\n--- 2. IMPROVED (20 GS + WLs + DWL + clean val) ---')
!python eval.py --name ptx-aug-train_NestedUNet_256_woDS --dataset ptx-test --img_ext .png --mask_ext .png
%cd /content

## Section 18 — TTA Evaluation

In [ ]:
%cd /content/Auto-Generate-WLs/pytorch-nested-unet
import os

tta_src = '''
import argparse, os
from glob import glob
import numpy as np, torch, yaml
import torch.backends.cudnn as cudnn
from albumentations import Resize, Normalize, Compose
from tqdm import tqdm
import archs
from dataset import Dataset

def main():
    p = argparse.ArgumentParser()
    p.add_argument("--name",     required=True)
    p.add_argument("--dataset",  required=True)
    p.add_argument("--img_ext",  default=".png")
    p.add_argument("--mask_ext", default=".png")
    p.add_argument("--tta",      default="h", choices=["none","h","hv"])
    args = vars(p.parse_args())
    with open(f"models/{args['name']}/config.yml") as f:
        cfg = yaml.load(f, Loader=yaml.FullLoader)
    cudnn.benchmark = True
    model = archs.__dict__[cfg["arch"]](
        num_classes=cfg["num_classes"],
        input_channels=cfg["input_channels"],
        deep_supervision=cfg["deep_supervision"]).cuda()
    model.load_state_dict(torch.load(f"models/{args['name']}/model.pth"))
    model.eval()
    ids = [os.path.splitext(os.path.basename(p))[0]
           for p in glob(f"inputs/{args['dataset']}/images/*{args['img_ext']}")]
    ds = Dataset(img_ids=ids,
        img_dir=f"inputs/{args['dataset']}/images",
        mask_dir=f"inputs/{args['dataset']}/masks",
        img_ext=args["img_ext"], mask_ext=args["mask_ext"],
        num_classes=cfg["num_classes"],
        transform=Compose([Resize(cfg["input_h"],cfg["input_w"]),Normalize()]))
    ious,dices,s = [],[],1e-5
    with torch.no_grad():
        for i in tqdm(range(len(ds))):
            img,gt,_ = ds[i]
            it = torch.from_numpy(img).unsqueeze(0).cuda()
            gt_t = torch.from_numpy(gt).unsqueeze(0).cuda()
            def fp(x):
                o=model(x); return torch.sigmoid(o[-1] if cfg["deep_supervision"] else o)
            p0 = fp(it)
            if args["tta"]=="none": pm=p0
            elif args["tta"]=="h":
                ph=fp(torch.flip(it,[3])); ph=torch.flip(ph,[3]); pm=(p0+ph)/2.
            else:
                ph=fp(torch.flip(it,[3])); ph=torch.flip(ph,[3])
                pv=fp(torch.flip(it,[2])); pv=torch.flip(pv,[2])
                phv=fp(torch.flip(it,[2,3])); phv=torch.flip(phv,[2,3]); pm=(p0+ph+pv+phv)/4.
            pb=pm>0.5; gb=gt_t>0.5
            ious.append(((pb&gb).sum().float()+s)/((pb|gb).sum().float()+s))
            dices.append((2.*(pm*gt_t).sum()+s)/(pm.sum()+gt_t.sum()+s))
    print(f"TTA={args['tta']}  IoU={np.mean([v.item() for v in ious]):.4f}  Dice={np.mean([v.item() for v in dices]):.4f}")

if __name__=="__main__": main()
'''

with open('eval_tta.py','w') as f: f.write(tta_src)

for mname,label in [
    ('ptx-gs-25_NestedUNet_256_woDS',     'BASELINE'),
    ('ptx-aug-train_NestedUNet_256_woDS', 'IMPROVED'),
]:
    for tta in ['none','h','hv']:
        print(f'\n=== {label} | TTA={tta} ===')
        os.system(f'python eval_tta.py --name {mname} --dataset ptx-test --tta {tta}')
%cd /content